# Running RFdiffusion on BU SCC

This guide uses your existing clone and environment. It covers the installation check,
manual repairs, a first GPU job, common design examples, and output inspection.
Commands that modify the installation are shown as **instructions**, not executable notebook cells.
No installation or job submission was performed when creating this notebook.

Open this notebook with any Python 3 Jupyter kernel on SCC (for example, through SCC OnDemand).
The code below calls SE3nv's Python explicitly, so SE3nv does **not** need its own Jupyter kernel.
Run cells in order. The submission cell defaults to preview only.


## 1. Installation findings — 2026-09-24

**Core inference packages are installed, but this environment is not ready for GPU inference.**

| Check | Result |
|---|---|
| Git revision | `613d80a`; tracked working tree clean before adding this notebook |
| Python | 3.9.25 |
| PyTorch | `1.9.1.post3`, Conda build `cpu_py39hc5866cc_3` |
| CUDA compiled into PyTorch | **None — CPU-only build** |
| DGL / e3nn | 0.9.1.post1 / 0.3.3; imports pass |
| SE3-Transformer / RFdiffusion | 1.0.0 / 1.1.0; inference imports pass |
| NumPy / SciPy | 1.24.3 / 1.10.1 |
| Hydra | 1.3.7 |
| Dependency metadata | `python -m pip check` passes |
| Inference entry point | `scripts/run_inference.py --help` passes |
| Checkpoints | All nine files loaded successfully on CPU; epoch-8 is named `Base_epoch8_ckpt.pt.1` |
| Optional W&B logging | `wandb` 0.12.0 import fails with protobuf 6.33.6 (descriptor error) |
| PPI scaffold examples / IGSO3 cache | Present |
| SE3nv notebook kernel | `ipykernel` is not installed; unnecessary for this guide's subprocess approach |

`torch.cuda.is_available() == False` can be normal on a login node, but
`torch.version.cuda is None` means this PyTorch installation cannot use a GPU even in a GPU job.
Installing `cudatoolkit` alone does not change a CPU-only PyTorch build.

The existing log at `outputs/2026-09-22/17-00-26/run_inference.log` reports CPU fallback
and reaches sequence initialization; it does not establish a completed design.
No new inference job was run during this inspection. The core imports and `pip check`
do not substitute for an end-to-end GPU test.

cd /project/dunlop/fereshteh/RFdiffusion/RFdiffusion/notebooks

hostname -f

python -m notebook --no-browser --ip="$(hostname -f)" --port=8888


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

REPO = Path('/project/dunlop/fereshteh/RFdiffusion/RFdiffusion')
ENV = Path('/projectnb/dunlop/fereshteh/.conda/envs/SE3nv')
PYTHON = ENV / 'bin/python'
MODELS = REPO / 'models'
assert (REPO / 'scripts/run_inference.py').is_file(), REPO
assert PYTHON.is_file(), PYTHON

# Avoid unrelated user-site packages and limit CPU use for inspection on login nodes.
check_env = dict(os.environ, PYTHONNOUSERSITE='1', PYTHONDONTWRITEBYTECODE='1',
                 DGLBACKEND='pytorch', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')

def check_python(source):
    result = subprocess.run([str(PYTHON), '-u', '-c', source], cwd=REPO,
                            env=check_env, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

print('Repository:', REPO)
print('Inference interpreter:', PYTHON)


In [ ]:
check_python("""
import torch, dgl, e3nn
from se3_transformer.model import SE3Transformer
from rfdiffusion.inference import model_runners
print('torch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU visible:', torch.cuda.is_available())
print('DGL:', dgl.__version__, 'e3nn:', e3nn.__version__)
print('Core imports passed.')
if torch.version.cuda is None:
    print('ACTION REQUIRED: replace CPU-only PyTorch before submitting GPU jobs.')
""")
result = subprocess.run([str(PYTHON), '-m', 'pip', 'check'], env=check_env,
                        text=True, capture_output=True)
print(result.stdout, result.stderr)
result.check_returncode()


In [ ]:
expected = ['Base_ckpt.pt', 'Complex_base_ckpt.pt', 'Complex_Fold_base_ckpt.pt',
            'InpaintSeq_ckpt.pt', 'InpaintSeq_Fold_ckpt.pt', 'ActiveSite_ckpt.pt',
            'Base_epoch8_ckpt.pt']
for name in expected:
    path = MODELS / name
    if path.is_file():
        print(f'{name}: {path.stat().st_size / 2**20:.1f} MiB')
    elif (MODELS / (name + '.1')).is_file():
        print(f'{name}: found as {name}.1; see the filename correction below')
    else:
        print(f'{name}: MISSING')


## 2. Manual repairs before GPU use

Run these commands yourself in an SCC **terminal**. They are not run by this notebook.

### Activate the existing environment

```bash
module load miniconda/23.11.0
source /share/pkg.8/miniconda/23.11.0/install/etc/profile.d/conda.sh
conda activate /projectnb/dunlop/fereshteh/.conda/envs/SE3nv
export PYTHONNOUSERSITE=1
cd /project/dunlop/fereshteh/RFdiffusion/RFdiffusion
```

### Replace CPU-only PyTorch with the matching CUDA build

The upstream environment uses PyTorch 1.9 and DGL for CUDA 11.1. Force the CUDA
PyTorch build rather than allowing Conda to choose another CPU build.
The installed torchvision 0.15.2 is also outside the matching PyTorch 1.9 family;
the command below requests torchvision 0.10.1 and torchaudio 0.9.1.
Preview the transaction first:

```bash
conda install --dry-run \
  --prefix /projectnb/dunlop/fereshteh/.conda/envs/SE3nv \
  --override-channels -c pytorch -c dglteam -c conda-forge -c defaults \
  'pytorch::pytorch=1.9.1=py3.9_cuda11.1_cudnn8.0.5_0' \
  'pytorch::torchvision=0.10.1' 'pytorch::torchaudio=0.9.1' \
  'cudatoolkit=11.1' 'dgl-cuda11.1=0.9.1post1' \
  'numpy<2' 'mkl<2024'
```

Review the proposed changes, ythen repeat that command **without `--dry-run`** to apply them.
This repair transaction has not been executed or solved during this audit; if Conda reports
a conflict, retain its error message for diagnosis instead of relaxing the CUDA build constraint.
Do not upgrade to a current PyTorch/DGL stack blindly: RFdiffusion's bundled SE3 code uses older APIs.
The version pairing follows the [PyTorch version table](https://github.com/pytorch/pytorch/wiki/PyTorch-Versions).

Afterward, repeat section 1. `torch.version.cuda` should report `11.1`.
GPU visibility still requires an SCC GPU allocation. A separate CUDA module is generally
unnecessary with the Conda CUDA runtime; loading one cannot fix CPU-only PyTorch.

### Correct the epoch-8 filename (optional for ordinary runs)

The standard unconditional and binder examples do not need this checkpoint.
If you want the upstream filename, run:

```bash
mv -n models/Base_epoch8_ckpt.pt.1 models/Base_epoch8_ckpt.pt
```

Alternatively, use the current `.pt.1` path with `inference.ckpt_override_path` when you
specifically want that model. No checkpoint was renamed during this audit.

### Optional: repair the W&B logging dependency

The separate `wandb` import fails with a protobuf descriptor error. The ordinary
RFdiffusion inference imports passed without W&B, so this is a secondary issue,
not the cause of the CPU-only PyTorch build. To use W&B or the bundled training utilities,
a compatibility repair to try in the activated environment is:

```bash
python -m pip install 'protobuf==3.20.3'
python -c 'import wandb; print(wandb.__version__)'
python -m pip check
```

These commands have not been applied. Recheck dependencies afterward.


## 3. Choose a first design

Start with one 50-residue unconditional backbone to verify the whole pipeline.
Use the default diffusion schedule; shortening it just for speed can change model behavior.
After this succeeds, try `150-150` and increase the design count.

Hydra overrides are individual command-line arguments. In shell commands, quote contig
strings, especially when they contain a space (`/0 ` is a chain break).
The Python list below already keeps each override together as one argument.


In [ ]:
OVERRIDES = [
    'contigmap.contigs=[50-50]',
    'inference.num_designs=1',
]
print(shlex.join([str(PYTHON), 'scripts/run_inference.py'] + OVERRIDES))


## 4. Prepare a GPU batch script

SCC uses `qsub` / Grid Engine. The cell below **previews** a script; it does not write or submit it.
It requests project `dunlop`, one P100 GPU, four CPU cores and two hours.
P100 is a conservative target for this legacy CUDA stack; V100 is another option after validation.
Check available models with `qgpus -s` in a terminal. Queue availability is not guaranteed.

The script activates SE3nv, fails if CUDA is unavailable, tests a DGL operation on the GPU,
then runs inference. It uses the scheduler's `CUDA_VISIBLE_DEVICES` assignment without overriding it.
Outputs go to a separate directory for every job, avoiding collisions on repeated runs.
See [BU's GPU job instructions](https://www.bu.edu/tech/support/research/software-and-programming/gpu-computing/).


In [ ]:
JOB_SCRIPT = REPO / 'notebooks' / 'run_rfdiffusion.qsub'
quoted_overrides = shlex.join(OVERRIDES)
batch_script = f"""#!/bin/bash -l
#$ -P dunlop
#$ -N rfdiffusion
#$ -cwd
#$ -j y
#$ -o {REPO}/outputs/scc_logs
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_type=P100
#$ -l h_rt=02:00:00

set -eo pipefail
module load miniconda/23.11.0
source /share/pkg.8/miniconda/23.11.0/install/etc/profile.d/conda.sh
conda activate {shlex.quote(str(ENV))}
export PYTHONNOUSERSITE=1
export DGLBACKEND=pytorch
export OMP_NUM_THREADS="${{NSLOTS:-1}}"
export OPENBLAS_NUM_THREADS="$OMP_NUM_THREADS"
export MKL_NUM_THREADS="$OMP_NUM_THREADS"
cd {shlex.quote(str(REPO))}

python -u - <<'PY'
import torch, dgl
import dgl.function as fn
assert torch.version.cuda is not None, 'CPU-only PyTorch: complete the repair first.'
assert torch.cuda.is_available(), 'No GPU visible in this allocation.'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, flush=True)
print('GPU:', torch.cuda.get_device_name(0), flush=True)
graph = dgl.graph(([0, 1], [1, 0])).to('cuda')
graph.ndata['x'] = torch.ones((2, 1), device='cuda')
graph.update_all(fn.copy_u('x', 'm'), fn.sum('m', 'y'))
assert torch.allclose(graph.ndata['y'], graph.ndata['x'])
torch.cuda.synchronize()
print('DGL GPU check passed.', flush=True)
PY

out_dir="{REPO}/outputs/scc_${{JOB_ID}}"
mkdir -p "$out_dir"
python -u scripts/run_inference.py \
    "inference.output_prefix=$out_dir/design" \
    {quoted_overrides}
"""
print(batch_script)


## 5. Save and submit when ready

Set `SUBMIT = True` in the next cell only after repairing and checking PyTorch.
The cell then creates the batch script and log directory and submits one job.
With the default `False`, it only displays the command.
If you change `OVERRIDES`, rerun the script-generation cell before submitting.

For terminal submission, save the preview as `notebooks/run_rfdiffusion.qsub`, then run:

```bash
cd /project/dunlop/fereshteh/RFdiffusion/RFdiffusion
mkdir -p outputs/scc_logs
qsub notebooks/run_rfdiffusion.qsub
```


In [ ]:
SUBMIT = False
if SUBMIT:
    # This test works on a login node: a CUDA build should still report a runtime version.
    check_python("import torch; assert torch.version.cuda is not None, 'Repair CPU-only PyTorch first'")
    (REPO / 'outputs' / 'scc_logs').mkdir(parents=True, exist_ok=True)
    JOB_SCRIPT.write_text(batch_script)
    subprocess.run(['bash', '-n', str(JOB_SCRIPT)], check=True)
    result = subprocess.run(['qsub', str(JOB_SCRIPT)], cwd=REPO,
                            text=True, capture_output=True)
    print(result.stdout, result.stderr)
    result.check_returncode()
else:
    print('Preview only:', shlex.join(['qsub', str(JOB_SCRIPT)]))
    print('Set SUBMIT = True to save the script and submit after the environment repair.')


## 6. Monitor and inspect results

Use the job number printed by `qsub`:

```bash
qstat -u "$USER"              # qw: queued; r: running; Eqw: scheduling error
qstat -j JOBID                # details while queued/running
qacct -j JOBID                # after completion; inspect failed and exit_status
tail -f outputs/scc_logs/rfdiffusion.oJOBID
```

Accounting may take a little time to appear. A job leaving `qstat` does not by itself prove success.
Expect `failed = 0`, `exit_status = 0`, and nonempty design files.
The first run can calculate IGSO3 tables; your clone already has a schedule cache,
but other settings may need new tables. Avoid launching many first-time jobs against
the same uncached schedule simultaneously.

Expected files in `outputs/scc_JOBID/`:

- `design_0.pdb`: generated backbone.
- `design_0.trb`: run configuration and residue-mapping metadata.
- `traj/`: intermediate diffusion structures, with trajectory output enabled.

Designed residues are typically glycine placeholders. RFdiffusion generates backbones;
sequence design (for example ProteinMPNN) and structure validation are separate steps.


In [ ]:
JOB_ID = ''  # Fill in the numeric ID returned by qsub.
if JOB_ID:
    assert JOB_ID.isdigit(), 'Use the numeric job ID.'
    output_dir = REPO / 'outputs' / f'scc_{JOB_ID}'
    print('Output directory:', output_dir)
    for path in sorted(output_dir.glob('*')):
        print(path.name, path.stat().st_size if path.is_file() else '<directory>')
    pdb = output_dir / 'design_0.pdb'
    if pdb.is_file():
        ca_count = sum(line.startswith('ATOM') and line[12:16].strip() == 'CA'
                       for line in pdb.read_text().splitlines())
        print('C-alpha atoms:', ca_count, '(expected 50 for the default first test)')
    else:
        print('No final PDB yet. Check the queue and batch log.')
else:
    print('Set JOB_ID after submitting your job.')


## 7. Other design tasks

Replace the `OVERRIDES` list in section 3 with one of these examples, then regenerate
the batch script. All relative paths are interpreted from `REPO`, the inner Git clone.

**Ten unconditional 150-residue backbones**

```python
OVERRIDES = ['contigmap.contigs=[150-150]', 'inference.num_designs=10']
```

**Motif scaffolding using the supplied RSV-F example**

```python
OVERRIDES = [
    'inference.input_pdb=examples/input_pdbs/5TPN.pdb',
    'contigmap.contigs=[10-40/A163-181/10-40]',
    'inference.num_designs=1',
]
```

`A163-181` preserves that input motif; each `10-40` segment generates a sampled number of
new residues. Chain letters and residue numbers refer to the PDB, not array indices.

**Binder design using the supplied insulin-receptor target**

```python
OVERRIDES = [
    'inference.input_pdb=examples/input_pdbs/insulin_target.pdb',
    'contigmap.contigs=[A1-150/0 70-100]',
    'ppi.hotspot_res=[A59,A83,A91]',
    'inference.num_designs=1',
    'denoiser.noise_scale_ca=0',
    'denoiser.noise_scale_frame=0',
]
```

The `/0 ` (including its space) separates target and binder chains. Hotspots specify
target residues to encourage the binder to contact. RFdiffusion selects the complex
checkpoint for this case. For your own target, change the PDB path, contig and hotspots
together. Start with one design before scaling; larger complexes can need more GPU memory and time.
See the repository's `examples/` and [README](../README.md) for partial diffusion,
symmetry, and fold-conditioned design.


## 8. Troubleshooting and references

| Symptom | What to check |
|---|---|
| `torch.version.cuda` is `None` | CPU-only PyTorch; follow section 2. A CUDA module will not fix this. |
| CUDA runtime exists, but GPU visibility is false | Check that the process is inside a GPU job and the scheduler assigned a GPU. |
| CUDA kernel or architecture error on a newer GPU | Use a validated P100/V100 allocation with this legacy environment, or validate a newer stack separately. |
| `ModuleNotFoundError` | Check `PYTHON` / `sys.executable`; activate the environment using its full path. |
| Hydra cannot parse a contig | Quote the whole argument in a shell; preserve the space after `/0`. |
| Missing checkpoint | Check spelling and `models/`; note the epoch-8 `.pt.1` filename. |
| GPU out of memory | Start with a shorter design/target crop, then consider a larger-memory compatible GPU. |
| `Eqw` or no batch log | Inspect `qstat -j JOBID`; ensure `outputs/scc_logs` exists before submitting. |
| No `.pdb` after a job finishes | Read the complete batch log and `qacct`; checkpoint loading alone is not success. |

Sources used for this guide:

- [Local upstream README](../README.md) and examples, at revision `613d80a`.
- [BU SCC GPU computing](https://www.bu.edu/tech/support/research/software-and-programming/gpu-computing/).
- [BU SCC batch submission](https://www.bu.edu/tech/support/research/system-usage/running-jobs/submitting-jobs/).
- [PyTorch previous versions](https://docs.pytorch.org/get-started/previous-versions/).
- [PyTorch version compatibility table](https://github.com/pytorch/pytorch/wiki/PyTorch-Versions).
